# タイタニック 生存予測 - 下調べ

SIGNATE 練習問題のデータを読み込んで中身を確認する。

## 1. データを読み込む

pandas をインポートして、ダウンロードした学習用データ・評価用データ・応募用サンプルファイルを読み込む。

In [ ]:
import pandas as pd

# index_col=0 : 1 列目（id）を「行の名前」として扱う指定。Python は 0 から数えるので 0 が 1 列目。
#               付けないと id が普通のデータ列になり、集計や学習の対象に混ざってしまう。
train = pd.read_csv("../data/train.csv", index_col=0)  # 学習用データ（survived あり）
test = pd.read_csv("../data/test.csv", index_col=0)  # 評価用データ（survived なし）

# header=None : このファイルは 1 行目が列名ではなくデータ。
#               付けないと 1 人目の予測結果が列名として消えてしまう。
sample_submit = pd.read_csv("../data/sample_submit.csv", index_col=0, header=None)  # 応募用サンプル

## 2. データの大きさを確認する

読み込んだ 3 つのデータが、それぞれ何行・何列あるかを表示する。

In [ ]:
# .shape は表の大きさを (行数, 列数) で返す
# print() で囲むのは 1 セルで 3 つ表示するため（囲まないと最後の 1 つしか出ない）
print("train:", train.shape)  # (445, 8) → 445 人 × 8 項目
print("test:", test.shape)  # (446, 7) → survived が無いので train より 1 列少ない
print("sample_submit:", sample_submit.shape)  # (446, 1) → 予測値の列だけ

# train と test の列数の差 1 が survived（助かったかどうか）。これを予測するのが今回の課題。
# id が列数に入っていないのは index_col=0 で行の名前にしたから（付けなければ 9 列と 8 列）。

## 3. 中身を先頭だけ見る

学習用データと評価用データの先頭 5 行を表示して、どんな列があり、どんな値が入っているかを確認する。

In [ ]:
# .head() は先頭 5 行だけ表示する。445 行すべて出すと画面が埋まるため。
#   head(3) で 3 行、tail() なら末尾 5 行
#   セルの最後に書いた値は自動で表示されるので print() は不要（表形式で見やすい）
train.head()

# 出力の見方
#   左端の太字が id。3, 4, 7... と飛ぶのは間の番号が test 側に振られているため
#   NaN は欠損（データが無い）
#   age が 35.0 と小数なのは、欠損を含む列を pandas が自動的に小数扱いするため。
#   年齢が小数という意味ではない

In [ ]:
test.head()

## 4. 列の情報と欠損を確認する

各列がどんな型で、値がいくつ入っているかを表示する。
ここで見つけた問題が、そのまま次の前処理でやることになる。

In [ ]:
# .info() は列ごとに「型」と「欠損でない値の個数」を一覧で出す
#   .head() が中身を数行見るのに対し、.info() は列全体の状態をまとめて見るためのもの
#   表を返すのではなくその場に直接書き出すので、print() は付けない
train.info()

# 出力の見方
#   Non-Null Count : 欠損していない値の個数。445 より少ない列は、その差だけ欠損がある
#   Dtype          : 値の型
#     int64   整数（例: pclass の 1, 2, 3）
#     float64 小数（例: fare の 53.1）
#     str     文字列。ここでは male / female のようなカテゴリ変数を指す
#             SIGNATE のチュートリアルでは object と表示されている。
#             pandas 3 で文字列専用の型ができたための違いで、中身は同じ

In [ ]:
test.info()

### 分かったこと

上の2つの出力から、前処理で対応が必要な点が3つ見つかる。

**1. `age` に欠損がある**（train 445 人中 360 人、test 446 人中 354 人しか値がない）

欠損のある行を捨てると学習データが 2 割近く減ってしまう。
何らかの値で埋めるか、列ごと使わないかを決める必要がある。

**2. `embarked` に欠損が 2 件ある**（train のみ。test には無い）

2 件だけなので影響は小さいが、放置するとモデルに渡せない。

**3. `sex` と `embarked` が文字列（`str` 型）**

モデルは数値しか扱えないため、`male` / `female` のような文字列は数値に変換する必要がある。
`pclass` が 1・2・3 という数値なのは、たまたま元データがそうなっていただけで、
これも本来はカテゴリ（客室の等級）である点に注意する。

なお `age` の型が `float64`（小数）になっているのは、3 で見たとおり、
欠損を含む数値列を pandas が自動的に小数として扱うため。